In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from example_models import get_linear_chain_2v
from mxlpy import scan, sensitivity

# Global sensitivity screening

Before committing to an expensive analysis &mdash; parameter fitting, profile-likelihood identifiability, or a full variance-based Sobol decomposition &mdash; it pays to ask a cheaper question first: **which parameters actually move the output at all?**

The **Morris method** answers exactly that. It is an *elementary-effects screening* method: it perturbs one parameter at a time along a handful of random trajectories through parameter space and measures how much the output changes. With only `n_trajectories * (k + 1)` model evaluations (`k` = number of parameters) it sorts parameters into three buckets:

| Reading                          | Interpretation                                  |
| -------------------------------- | ----------------------------------------------- |
| low `mu_star`                    | unimportant &mdash; safe to fix to a constant   |
| high `mu_star`, low `sigma`      | important and roughly **linear**                |
| high `mu_star`, high `sigma`     | important and **nonlinear** or interacting      |

where `mu_star` is the mean *absolute* elementary effect and `sigma` its standard deviation across trajectories.

## The output function

Sensitivity is always measured with respect to *some scalar output* of the model. Rather than ship a fixed menu of reductions, `sensitivity.morris` asks you for an **output function**:

```python
output(model, samples: pd.DataFrame) -> np.ndarray
```

It receives the model and the full matrix of parameter samples (one row per sample, one column per parameter) and returns one scalar per row. The natural way to write it is to lean on [`mxlpy.scan`](scans.ipynb), which already evaluates the whole sample matrix in parallel, caches results, and &mdash; importantly &mdash; returns `NaN` for any sample whose integration failed. That `NaN` flows straight through to the sensitivity indices, so failed runs surface honestly instead of being silently masked.

Any reduction `scan` can express is available for free:

- **steady-state value** of a species: `scan.steady_state(model, to_scan=samples).variables[name]`
- **peak value** over a time course: `scan.time_course(model, to_scan=samples, time_points=...).get_agg_per_run("max")[name]`
- **area under the curve**: pass `np.trapezoid` as the aggregator to `get_agg_per_run`

Here we screen the steady-state concentration of `x` in a simple linear chain $\varnothing \xrightarrow{v_1} x \xrightarrow{v_2} y \xrightarrow{v_3} \varnothing$ against its three rate constants.

In [ ]:
def steady_state_x(model, samples):
    return scan.steady_state(model, to_scan=samples).variables["x"].to_numpy()


result = sensitivity.morris(
    get_linear_chain_2v(),
    output=steady_state_x,
    param_bounds={"k1": (0.5, 2.0), "k2": (0.5, 2.0), "k_out": (0.5, 2.0)},
    n_trajectories=20,
    seed=0,
)
result.sort_values("mu_star", ascending=False)

The result is a labelled `pandas.DataFrame` indexed by parameter name with columns `mu`, `mu_star`, `sigma`, and `mu_star_conf` (a bootstrap confidence interval on `mu_star`).

The reading is immediate: at steady state $x = k_1 / k_2$, so `k1` and `k2` dominate while `k_out` &mdash; which only drains the downstream species `y` &mdash; has an elementary effect of essentially zero. A screening run like this lets you drop `k_out` from a subsequent fit without losing anything.

## Visualising the screen

Two plots make the screen easy to read: a ranking of `mu_star` (with its confidence interval), and a `mu_star`-vs-`sigma` scatter that separates linear from nonlinear influence.

In [ ]:
ranked = result.sort_values("mu_star")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))

ax1.barh(ranked.index, ranked["mu_star"], xerr=ranked["mu_star_conf"])
ax1.set(xlabel=r"$\mu^*$ (mean absolute effect)", title="Parameter ranking")

ax2.scatter(result["mu_star"], result["sigma"])
for name, row in result.iterrows():
    ax2.annotate(name, (row["mu_star"], row["sigma"]))
ax2.set(xlabel=r"$\mu^*$", ylabel=r"$\sigma$", title="Linear vs. nonlinear")

fig.tight_layout()
plt.show()

## Cost and reproducibility

Morris is cheap: with `n_trajectories=20` and `k=3` parameters the screen above used `20 * (3 + 1) = 80` model evaluations &mdash; a tiny fraction of what a variance-based method would need. That is the point: **screen first with Morris, then quantify the survivors** with a more expensive method.

Pass a `seed` for reproducible sampling and bootstrap confidence intervals, and tune `num_levels` (default `4`) to control the granularity of the sampling grid.

In [ ]:
n_params = 3
n_trajectories = 20
print(f"Evaluations: {n_trajectories * (n_params + 1)}")